In [65]:
!rm functions.py
!rm pretext_tasks.py
!pwd
!ls
from google.colab import files
uploaded = files.upload()

rm: cannot remove 'functions.py': No such file or directory
rm: cannot remove 'pretext_tasks.py': No such file or directory
/content
 data  'functions (1).py'   __pycache__   sample_data


Saving functions.py to functions.py
Saving pretext_tasks.py to pretext_tasks.py


In [66]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim
from torch import nn
import os
import sys
import shutil
import datetime
from tqdm import tqdm
import argparse
import matplotlib.pyplot as plt
import numpy as np
import random
from torchvision.transforms.functional import to_pil_image, to_tensor
import pretext_tasks
import importlib
importlib.reload(pretext_tasks)

from functions import (PIRL, AverageMeter)
from pretext_tasks import (Mirror_Detection, Grayscale_Colorization, Grayscale_Colorization_Batch, Rotate, Jigsaw, Jigsaw_Batch)

In [67]:
# Utility functions
def create_save_dir():
    save_dir = os.path.join("./checkpoints", datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(save_dir, exist_ok=True)
    return save_dir

def save_checkpoint(state, is_best, save_dir):
    torch.save(state, os.path.join(save_dir, 'checkpoint.pth'))
    if is_best:
        shutil.copyfile(os.path.join(save_dir, 'checkpoint.pth'), os.path.join(save_dir, 'best_model.pth'))

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['state_dict'])
    if optimizer and 'optimizer' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch = checkpoint.get('epoch', 0) + 1
    print(f"Loaded checkpoint from '{path}' (epoch {start_epoch})")
    return start_epoch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def imshow(img, mean=(0.5,0.5,0.5), std=(1.0,1.0,1.0)):
    img = img.permute(1, 2, 0).cpu().numpy()
    img = img * std + mean
    img = img.clip(0, 1)
    return img

def visualize_predictions(model, loader, device, classes):
    model.eval()
    dataiter = iter(loader)
    images, labels = next(dataiter)
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    if isinstance(outputs, tuple):
      outputs = outputs[0]

    _, predicted = torch.max(outputs, 1)

    # Show 4 images
    fig, axs = plt.subplots(2, 2, figsize=(8, 8))
    for idx, ax in enumerate(axs.flatten()):
        ax.imshow(imshow(images[idx]))
        ax.set_title(f"Predicted: {classes[predicted[idx]]}\nActual: {classes[labels[idx]]}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def train_one_epoch(model, loader, criterion, optimizer, epoch, device, pretext_task = "None"):
    model.train()
    losses = AverageMeter()

    if pretext_task == "None":
      print("no pretask")

    for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
        images, labels = images.to(device), labels.to(device)

        if pretext_task == "Mirror":
          images, transformed_images = Mirror_Detection(flip_prob=0.5, return_image=True)(images, alwaysFlip = True)
        elif pretext_task == "Grayscale":
          images, transformed_images = Grayscale_Colorization_Batch(return_image=True)(images)
        elif pretext_task == "Rotate":
          images, transformed_images = Rotate(return_image=True)(images)
        elif pretext_task == "Jigsaw":
          # apply JigsawBatch to entire batch at once
          jigsaw_batch = Jigsaw_Batch(return_image=True)
          images, transformed_images = jigsaw_batch(images)
          images, transformed_images = images.to(device), transformed_images.to(device)
        elif pretext_task == "GrayscaleRotate":
          # Grayscale
          images, gray_images = Grayscale_Colorization_Batch(return_image=True)(images)
          # Rotate the grayscale images
          _, transformed_images = Rotate(return_image=True)(gray_images)

        else: #no pretext task
          transformed_images = images

        # Forward pass for both original and flipped images
        (outputs_logits, _), (transformed_logits, _) = model(images, transformed_x=transformed_images)

        optimizer.zero_grad()
        loss = criterion(outputs_logits, transformed_logits, labels)

        loss.backward()
        optimizer.step()
        losses.update(loss.item(), images.size(0))

    print(f"Epoch {epoch} - Training Loss: {losses.avg:.4f}")

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs, _ = model(images)  # Only take the logits (ignore features)
            _, predicted = torch.max(outputs.data, 1)  # Get the predicted class
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100.0 * correct / total
    print(f"Validation Accuracy: {accuracy:.2f}%")
    return accuracy

In [59]:
# No Pretext Task
# Data transformations
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (1.0, 1.0, 1.0)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (1.0, 1.0, 1.0))
])
# CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_original = PIRL().to(device)
criterion = model_original.loss
optimizer = optim.SGD(model_original.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_original, trainloader, criterion, optimizer, epoch, device, pretext_task = "None")
    acc = evaluate(model_original, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_original, testloader, device, classes)

Using device: cuda
no pretask


Epoch 1: 100%|██████████| 782/782 [01:49<00:00,  7.11it/s]

Epoch 1 - Training Loss: 2.5717


Validation Accuracy: 29.81%
no pretask


Epoch 2: 100%|██████████| 782/782 [01:49<00:00,  7.11it/s]

...

Epoch 8 - Training Loss: 1.1096


Validation Accuracy: 56.97%
no pretask


Epoch 9: 100%|██████████| 782/782 [01:50<00:00,  7.11it/s]

Epoch 9 - Training Loss: 1.0694


Validation Accuracy: 63.96%


In [72]:
# Mirror Pretext Task
# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_mirror = PIRL().to(device)
criterion = model_mirror.loss
optimizer = optim.SGD(model_mirror.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_mirror, trainloader, criterion, optimizer, epoch, device, pretext_task = "Mirror")
    acc = evaluate(model_mirror, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_mirror, testloader, device, classes)

Using device: cuda


Epoch 1: 100%|██████████| 782/782 [01:50<00:00,  7.06it/s]

Epoch 1 - Training Loss: 2.2034


Validation Accuracy: 33.45%


Epoch 2: 100%|██████████| 782/782 [01:50<00:00,  7.10it/s]

...

Epoch 8 - Training Loss: 1.1416


Validation Accuracy: 58.26%


Epoch 9: 100%|██████████| 782/782 [01:49<00:00,  7.11it/s]

Epoch 9 - Training Loss: 1.0844


Validation Accuracy: 62.94%


In [ ]:
# Grayscale Pretext Task
# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_grayscale = PIRL().to(device)
criterion = model_grayscale.loss
optimizer = optim.SGD(model_grayscale.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_grayscale, trainloader, criterion, optimizer, epoch, device, pretext_task = "Grayscale")
    acc = evaluate(model_grayscale, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_grayscale, testloader, device, classes)

Using device: cuda


Epoch 1: 100%|██████████| 782/782 [01:50<00:00,  7.10it/s]

Epoch 1 - Training Loss: 2.3242


Validation Accuracy: 33.55%


Epoch 2:  88%|████████▊ | 690/782 [01:37<00:12,  7.25it/s]

In [53]:
# Jigsaw Pretext Task
# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_jigsaw = PIRL().to(device)
criterion = model_jigsaw.loss
optimizer = optim.SGD(model_jigsaw.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_jigsaw, trainloader, criterion, optimizer, epoch, device, pretext_task = "Jigsaw")
    acc = evaluate(model_jigsaw, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_jigsaw, testloader, device, classes)

Using device: cuda


Epoch 1: 100%|██████████| 782/782 [02:08<00:00,  6.08it/s]

Epoch 1 - Training Loss: 2.4730


Validation Accuracy: 13.67%


Epoch 2: 100%|██████████| 782/782 [02:09<00:00,  6.05it/s]

...

Epoch 8 - Training Loss: 1.4064


Validation Accuracy: 16.52%


Epoch 9: 100%|██████████| 782/782 [02:09<00:00,  6.04it/s]

Epoch 9 - Training Loss: 1.3714


Validation Accuracy: 12.44%


In [70]:
# Rotate Pretext Task
# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_rotate = PIRL().to(device)
criterion = model_rotate.loss
optimizer = optim.SGD(model_rotate.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_rotate, trainloader, criterion, optimizer, epoch, device, pretext_task = "Rotate")
    acc = evaluate(model_rotate, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_rotate, testloader, device, classes)

Using device: cuda


Epoch 1: 100%|██████████| 782/782 [01:51<00:00,  7.04it/s]

Epoch 1 - Training Loss: 2.1979


Validation Accuracy: 36.70%


Epoch 2: 100%|██████████| 782/782 [01:50<00:00,  7.07it/s]

...

Epoch 8 - Training Loss: 1.1282


Validation Accuracy: 62.65%


Epoch 9: 100%|██████████| 782/782 [01:50<00:00,  7.08it/s]

Epoch 9 - Training Loss: 1.0953


Validation Accuracy: 58.99%


In [69]:
# GrayscaleRotate Pretext Task
# Model, loss, optimizer
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_grayscaleRotate = PIRL().to(device)
criterion = model_grayscaleRotate.loss
optimizer = optim.SGD(model_grayscaleRotate.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

best_acc = 0
start_epoch = 1

for epoch in range(1, 10):
    train_one_epoch(model_grayscaleRotate, trainloader, criterion, optimizer, epoch, device, pretext_task = "GrayscaleRotate")
    acc = evaluate(model_grayscaleRotate, testloader, device)
    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    if epoch % 2 == 0:
        classes = trainset.classes
        #visualize_predictions(model_grayscaleRotate, testloader, device, classes)

Using device: cuda


Epoch 1: 100%|██████████| 782/782 [01:50<00:00,  7.06it/s]

Epoch 1 - Training Loss: 2.2527


Validation Accuracy: 31.40%


Epoch 2: 100%|██████████| 782/782 [01:50<00:00,  7.06it/s]

...

Epoch 8 - Training Loss: 1.1447


Validation Accuracy: 59.41%


Epoch 9: 100%|██████████| 782/782 [01:50<00:00,  7.06it/s]

Epoch 9 - Training Loss: 1.1133


Validation Accuracy: 64.04%
